# AutoGluon
Matar una mosca con una  bazooka, entrena automaticamente modelos de:


*   Estadistica Clasica
*   Machine Learning ~ GBDT
*   Deep Learning


 ['SeasonalNaive', 'RecursiveTabular', 'DirectTabular', 'DynamicOptimizedTheta', 'Chronos2', 'Chronos2SmallFineTuned', 'AutoETS', 'ChronosWithRegressor[bolt_small]', 'TemporalFusionTransformer', 'DeepAR']



![Matar una mosca con una bazooka ](https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/Bazooka_fly.jpg)

## 0.1 Init ambiente Google Colab

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [1]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


Para correr la siguiente celda es fundamental, POR UNICA VEZ, seguir los siguientes pasos

* Registrar usuario en Kaggle con la cuenta de email de la Universidad Austral
* Hacer el "Join Competition"  a la competencia de  Labo 3
* Generar el archivo kaggle.json  a partir de   https://www.kaggle.com/settings/account  y presione  "Create Legacy API Key"
* Crear carpeta  labo3  en  el Google Drive
* Dentro de la carpeta labo3 crear carpeta   kaggle
* Subir a la carpeta kaggle el archivo  kaggle.json


In [2]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp "/content/buckets/b1/kaggle/kaggle (2).json"  ~/.kaggle/kaggle.json
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"


# 1  Modelo AutoGluon

## 1.1 Init Experimento

In [3]:
# instalacion de paquetes que NO vienen por default en Colab
# autogluon es MUY pesado. varios minutos se perderan aqui
!pip install uv
!uv pip install -q kaggle
!uv pip install autogluon[all]


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.3/26.3 MB 55.2 MB/s eta 0:00:00
Using Python 3.12.13 environment at: /usr
Resolved 253 packages in 4.24s
Prepared 84 packages in 2m 00s
Uninstalled 9 packages in 1.39s
Installed 84 packages in 849ms
 + adagio==0.2.6
 + aiohttp-cors==0.8.1
 + alembic==1.18.5
 + autogluon==1.5.0
 + autogluon-common==1.5.0
 + autogluon-core==1.5.0
 + autogluon-features==1.5.0
 + autogluon-multimodal==1.5.0
 + autogluon-tabular==1.5.0
 + autogluon-timeseries==1.5.0
 + boto3==1.43.39
 + botocore==1.43.39
 + catboost==1.2.10
 + chronos-forecasting==2.3.0
 + colorama==0.4.6
 + colorful==0.5.8
 + colorlog==6.10.1
 + coreforecast==0.0.16
 + distlib==0.4.3
 + einx==0.4.3
 + evaluate==0.4.6
 + fugue==0.9.7
 + gluonts==0.16.3
 - huggingface-hub==1.20.1
 + huggingface-hub==0.36.2
 + jmespath==1.1.0
 - jsonschema==4.26.0
 + jsonschema==4.23.0
 + lightning==2.5.6
 + lightning-utilities==0.15.3
 + loguru==0.7.3
 + mako==1.3.12
 + mlforecast==0.14.0
 + model-index==0.1.11


In [4]:
# funcion para hacer submits a Kaggle
def kaggle_submit(competencia, archivo, mensaje):

  # comando
  comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
  # ejecucion
  os.system(comando)


In [5]:
import os as os
import numpy as np
import polars as pl

from datetime import datetime
from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame

Por favor, cargar aqui SU semilla primigenia
<br> **Muy importante**, cambiar el numero de experimento en cada corrida. Usted ha sido notificado !
<br> Si cada corrida no está en una nueva carpeta virgen, entonces se reutilizarn modelos viejos corridos con otros parametros
https://www.youtube.com/shorts/0_0Kzqpdn1o

In [6]:
# defino los parametros
PARAM = {'experimento':'AutoGluon-01',
  'kaggle_competition':'labo-iii-2026-ba',
  'semilla_primigenia':199410
}

In [7]:
# creo la carpeta del experimento y hago el chdir
ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
print(ruta)
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

/content/buckets/b1/exp/AutoGluon-01


## 1.2 Init AutoGluon

In [8]:
# cargo el dataset del sell-in
dataset = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator="\t")

In [9]:
# agrupo por product_id, periodo
tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
)

tb_ventas = tb_ventas.sort(["product_id", "periodo"])

In [10]:
# cargo la tabla "apredecir" que contiene los 780 productos que deben predecirse las ventas de 202002
tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator="\t")


# Filtro tb_ventas a solo las que debo predecir
print(tb_ventas.height)
tb_ventas = tb_ventas.join(tb_apredecir,
  on="product_id",
  how="inner"
)
print(tb_ventas.height)
tb_ventas = tb_ventas.sort(["product_id", "periodo"])

31243
22349


In [11]:
# paso de periodo a  timestamp
tb_ventas = tb_ventas.with_columns(
    (pl.col('periodo').cast(pl.String).str.to_datetime('%Y%m')).alias('timestamp')
)

Opcion de *empiojar el dataset*
<br>agregando ruido relativo a las ventas
<br>Un Experimento no se le niega a nadie

In [12]:
empiojar= False
empiojar_ruido= 0.25

if empiojar:
  np.random.seed(PARAM['semilla_primigenia'])
  tb_ventas = tb_ventas.sort(["product_id", "periodo"])
  # vector con el ruido multiplicativo de media 1.0  y desvio  'empiojar_ruido'
  noise_multiplier = np.random.lognormal(mean=0.0, sigma=empiojar_ruido, size=tb_ventas.height)

  tb_ventas = tb_ventas.with_columns(
    (pl.col("tn") * pl.lit(noise_multiplier)).alias("tn")
  )


## 1.3 Entrenamiento AutoGluon

La magia del Auto Machine Learning  aplicada a un dataset que posee MULTIPLES series de tiempo que se analizan en forma conjunta, suponiendo algun tipo de correlacion entre ellas.

AutoGluon TimeSeriesPredictor predicts future values of **multiple related** time series.

TimeSeriesPredictor provides probabilistic (quantile) multi-step-ahead orecasts for univariate time series. The forecast includes both the mean (i.e.,
 onditional expectation of future values given the past), as well as the quantiles of the forecast distribution, indicating the range of possible future outcomes.

TimeSeriesPredictor fits both “global” deep learning models that are **shared across all time series** (e.g., DeepAR, Transformer), as well as “local”
 statistical models that are fit to each individual time series (e.g., ARIMA, ETS).

TimeSeriesPredictor expects input data and makes predictions in the TimeSeriesDataFrame format.


https://auto.gluon.ai/stable/api/autogluon.timeseries.TimeSeriesPredictor.html

In [13]:
# paso a formato  ts = time_series
ts_data = TimeSeriesDataFrame.from_data_frame(
  tb_ventas.to_pandas(), # tristemente AutoGluon espera un dataframe Pandas ...
  timestamp_column='timestamp',
  id_column='product_id'
)

Elegir cuidadosamente la metrica a utilizar
<br>Probar alternativas !
<br>Ese celda lleva 30 minutos en correr
<br>https://auto.gluon.ai/stable/tutorials/timeseries/forecasting-metrics.html#forecasting-metrics
<br>https://auto.gluon.ai/stable/api/autogluon.timeseries.TimeSeriesPredictor.fit.html#autogluon.timeseries.TimeSeriesPredictor.fit
<br>https://auto.gluon.ai/stable/api/autogluon.timeseries.TimeSeriesPredictor.html

In [14]:
# Entrenamiento, esta es la parte pesada
global_eval_metric = 'RMSE' # alternativa RMSE

# defino
modelo = TimeSeriesPredictor(
  prediction_length= 2,  # horizonte de prediccion
  target= 'tn',
  freq= 'MS',  # Frecuencia mensual (Month Start)
  eval_metric= global_eval_metric
)

# entreno, le tira con muchisimos algorimtos, y prueba ensamblarlos
modelo.fit(ts_data,
  num_val_windows= 2,
  time_limit= 3600, # una hora
  presets= "best_quality",  # Máxima calidad posible, obvio mayor tiempo de corrida
  random_seed= PARAM['semilla_primigenia']
)

Beginning AutoGluon training... Time limit = 3600s
AutoGluon will save models to '/content/.drive/My Drive/labo3/exp/AutoGluon-01/AutogluonModels/ag-20260701_222825'
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
GPU Memory:         
Total GPU Memory:   Free: 0.00 GB, Allocated: 0.00 GB, Total: 0.00 GB
GPU Count:          0
Memory Avail:       11.24 GB / 12.67 GB (88.7%)
Disk Space Avail:   76.14 GB / 107.72 GB (70.7%)
Setting presets to: best_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': RMSE,
 'freq': 'MS',
 'hyperparameters': 'default',
 'known_covariates_names': [],
 'num_val_windows': 2,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed':

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/478M [00:00<?, ?B/s]

	-33.2467      = Validation score (-RMSE)
	57.33   s     = Training runtime
	35.65   s     = Validation (prediction) runtime
Training timeseries model Chronos2SmallFineTuned. Training for up to 577.3s of the 3463.7s of remaining time.


config.json:   0%|          | 0.00/969 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/112M [00:00<?, ?B/s]

	-33.3568      = Validation score (-RMSE)
	554.59  s     = Training runtime
	8.91    s     = Validation (prediction) runtime
Training timeseries model AutoETS. Training for up to 580.0s of the 2900.1s of remaining time.
	-36.9451      = Validation score (-RMSE)
	23.13   s     = Training runtime
	24.17   s     = Validation (prediction) runtime
Training timeseries model ChronosWithRegressor[bolt_small]. Training for up to 750.9s of the 2852.7s of remaining time.
	Skipping covariate_regressor since the dataset contains no known_covariates or static_features.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/191M [00:00<?, ?B/s]

	Skipping covariate_regressor since the dataset contains no known_covariates or static_features.
	-34.7490      = Validation score (-RMSE)
	6.71    s     = Training runtime
	3.36    s     = Validation (prediction) runtime
Training timeseries model TemporalFusionTransformer. Training for up to 1121.3s of the 2842.5s of remaining time.
	-35.6964      = Validation score (-RMSE)
	425.19  s     = Training runtime
	0.89    s     = Validation (prediction) runtime
Training timeseries model DeepAR. Training for up to 1816.3s of the 2416.3s of remaining time.
	-33.4605      = Validation score (-RMSE)
	214.04  s     = Training runtime
	2.04    s     = Validation (prediction) runtime
Fitting 1 ensemble(s), in 1 layers.
Training ensemble model WeightedEnsemble. Training for up to 2199.9s.
	Ensemble weights: {'AutoETS': 0.21, 'Chronos2': 0.01, 'DeepAR': 0.37, 'SeasonalNaive': 0.41}
	-28.5403      = Validation score (-RMSE)
	1.30    s     = Training runtime
	63.49   s     = Validation (prediction) ru

## 1.4 Prediccion AutoGluon

In [15]:
# predict a partir los mismos datos

tb_forecast = modelo.predict(ts_data,
  random_seed= PARAM['semilla_primigenia']
)

display(tb_forecast)

data with frequency 'IRREG' has been resampled to frequency 'MS'.
Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble


mean         0.1         0.2          0.3  \
item_id timestamp                                                      
20001   2020-01-01  1234.189143  788.713309  961.976291  1060.388779   
        2020-02-01  1216.577553  779.029437  944.013843  1049.376888   
20002   2020-01-01  1175.920070  780.889943  922.292188  1020.624706   
        2020-02-01  1055.687724  634.106210  785.977574   899.591542   
20003   2020-01-01   813.663971  562.245312  653.162582   716.464326   
...                         ...         ...         ...          ...   
21266   2020-02-01     0.051489   -0.085949   -0.038607    -0.005093   
21267   2020-01-01     0.028510   -0.033840   -0.011416     0.003649   
        2020-02-01     0.025317   -0.060693   -0.031456    -0.008901   
21276   2020-01-01     0.012627   -0.005584    0.000707     0.005009   
        2020-02-01     0.011516   -0.015362   -0.005988     0.000599   

                            0.4          0.5          0.6          0.7  \
item_id timestamp                                                        
20001   2020-01-01  1147.767828  1225.798480  1316.000997  1405.613706   
        2020-02-01  1137.481188  1220.925942  1304.927870  1389.869999   
20002   2020-01-01  1104.495436  1176.181702  1249.755453  1325.829922   
        2020-02-01   985.042952  1057.624995  1136.969120  1219.314209   
20003   2020-01-01   766.478606   817.802330   862.446575   915.618302   
...                         ...          ...          ...          ...   
21266   2020-02-01     0.024046     0.051626     0.078721     0.108041   
21267   2020-01-01     0.016764     0.028958     0.040822     0.053825   
        2020-02-01     0.009048     0.025804     0.042534     0.060246   
21276   2020-01-01     0.008957     0.012599     0.016171     0.019929   
        2020-02-01     0.006448     0.011828     0.016894     0.022306   

                            0.8          0.9  
item_id timestamp                             
20001   2020-01-01  1510.093294  1671.194548  
        2020-02-01  1506.122264  1635.243193  
20002   2020-01-01  1427.105632  1574.478311  
        2020-02-01  1317.132021  1465.019271  
20003   2020-01-01   980.970598  1068.990603  
...                         ...          ...  
21266   2020-02-01     0.142677     0.190708  
21267   2020-01-01     0.069602     0.090770  
        2020-02-01     0.081539     0.111006  
21276   2020-01-01     0.024597     0.031711  
        2020-02-01     0.029114     0.038540  

[1560 rows x 10 columns]

In [16]:
# paso a formato Polars, teniendo en cuenta el indice
tb_forecast = pl.from_pandas(tb_forecast.reset_index())

In [17]:
# me quedo con ls predicciones de febrero-2020
# en 'mean' esta la prediccion, mas alla de los n-tiles
tb_final = tb_forecast.filter(pl.col("timestamp") ==  datetime(2020, 2, 1)).select(["item_id","mean"])

display(tb_final)

item_id,mean
i64,f64
20001,1216.577553
20002,1055.687724
20003,670.566526
20004,493.249225
20005,488.540964
…,…
21263,0.039609
21265,0.048906
21266,0.051489


## 1.5 Submit a Kaggle

In [18]:
# cambio nombre de campos a los que reconoce Kaggle
tb_final = tb_final.rename({
  "item_id": "product_id",
  "mean": "tn",
})

In [19]:
# Submit a Kaggle
if not empiojar:
  archivo= "AutoGluon_" + global_eval_metric + ".csv"
  mensaje= "AutoGluon " + global_eval_metric
else:
  archivo= "AutoGluon_empiojado_" + global_eval_metric + ".csv"
  mensaje= "AutoGluon logEMPIOJADO " + global_eval_metric + " al " + str(empiojar_ruido)

tb_final.write_csv(archivo)

kaggle_submit(PARAM['kaggle_competition'], archivo, mensaje )